### structured response format 

In [ ]:
# --- Structured output with Pydantic (NOTES.md §9) ---
# Use Pydantic when data crosses a trust boundary (LLM/user/API) — it validates and
# coerces the model's text (e.g. "23" -> 23.0), and each Field(description=...) is fed
# into the schema/prompt so the model knows what to fill in.
# pydantic
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
model=init_chat_model("ollama:qwen3:8b")

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="Name of the director of the movie")
    rating:float=Field(description="The movie's rating out of 10")

# with_structured_output makes the model return a validated Movie instance, not free text.
model_with_structure=model.with_structured_output(Movie)
model_with_structure.invoke("Provide details about movie Inception")

In [2]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="Name of the director of the movie")
    rating:float=Field(...,description="The movie's rating out of 10")

model_with_structure=model.with_structured_output(Movie, include_raw=True)
model_with_structure.invoke("Provide details about movie Inception")

{'raw': AIMessage(content='{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.885714285714286}\n\n  \t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t ', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-06-10T08:27:57.3148239Z', 'done': True, 'done_reason': 'stop', 'total_duration': 73821302200, 'load_duration': 10253281700, 'prompt_eval_count': 744, 'prompt_eval_duration': 1152595000, 'eval_count': 51, 'eval_duration': 4185984000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019eb0a4-13a4-75c3-905e-f5359f7907c1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 744, 'output_tokens': 51, 'total_tokens': 795}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.885714285714286),
 'parsing_error': None}

In [4]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None =Field(None, description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)
model_with_structure.invoke("Provide details about movie Inception")

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Yusuf'), Actor(name='Emily Watson', role='Mal'), Actor(name='Michael Caine', role='Professor Fischer')], genres=['Science Fiction', 'Action', 'Thriller'], budget=0.0)